Tuesday 24<sup>th</sup> February 2026

So in this session we will created a database and add the details in the csv file data in the tables.

In [1]:
import duckdb

print("duck db version",duckdb.__version__)

duck db version 1.4.1


In [6]:
conn=duckdb.connect()
result=conn.sql("SELECT 42 AS answer,'Hello Emmanuel' AS greetings")
print(result)
'''
df=result.df
print(df)

'''

┌────────┬────────────────┐
│ answer │   greetings    │
│ int32  │    varchar     │
├────────┼────────────────┤
│     42 │ Hello Emmanuel │
└────────┴────────────────┘



'\ndf=result.df\nprint(df)\n\n'

In [7]:
data = [
    {"site": "Laisamis", "date": "2023-01-01", "ghi": 6.8, "temp": 28.5},
    {"site": "Narok",    "date": "2023-01-01", "ghi": 6.2, "temp": 26.1},
    {"site": "Laisamis", "date": "2023-01-02", "ghi": 7.1, "temp": 29.2},
    {"site": "Nairobi",  "date": "2023-01-01", "ghi": 5.9, "temp": 25.8},
    {"site": "Narok",    "date": "2023-01-02", "ghi": 6.5, "temp": 27.0},
    {"site": "Laisamis", "date": "2023-01-03", "ghi": 4.8, "temp": 24.5},  # cloudy day example
]

conn.register("energy_readings", data)

InvalidInputException: Invalid Input Error: Python Object "energy_readings" of type "list" not suitable for replacement scans.
Make sure that "energy_readings" is either a pandas.DataFrame, duckdb.DuckDBPyRelation, pyarrow Table, Dataset, RecordBatchReader, Scanner, or NumPy ndarrays with supported format

In [2]:
result=conn.sql("SELECT * FROM read_csv_auto('power_generation_in kenya2.csv')")
result_df=result.df
print(result_df)

NameError: name 'conn' is not defined

In [3]:
conn.sql("""
SELECT date, series, generation_twh
FROM read_csv_auto('power_generation_in kenya2.csv')
WHERE series='Total generation'
""").show()

NameError: name 'conn' is not defined

In [ ]:
conn.sql("""
SELECT date, generation_twh
FROM read_csv_auto('power_generation_in kenya2.csv')
WHERE series='Total generation' 
ORDER BY generation_twh DESC
LIMIT 10
""").show()

┌─────────┬────────────────┐
│  date   │ generation_twh │
│ varchar │     double     │
├─────────┼────────────────┤
│ 2023-07 │           1.12 │
│ 2023-08 │            1.1 │
│ 2022-10 │            1.1 │
│ 2022-07 │            1.1 │
│ 2024-10 │            1.1 │
│ 2022-08 │            1.1 │
│ 2024-07 │            1.1 │
│ 2022-09 │           1.08 │
│ 2024-08 │           1.08 │
│ 2022-12 │           1.08 │
├─────────┴────────────────┤
│ 10 rows        2 columns │
└──────────────────────────┘



In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
#new connection to duckdb database as power_gen_conn
power_gen_conn=duckdb.connect("power_generation.db")


In [7]:
power_gen_conn.sql("CREATE TABLE IF NOT EXISTS energy_mix AS SELECT * FROM read_csv_auto('power_generation_in kenya2.csv')")

In [8]:
power_gen_conn.sql("SELECT * FROM energy_mix").show()

┌─────────┬──────────────────┬────────────────┐
│  date   │      series      │ generation_twh │
│ varchar │     varchar      │     double     │
├─────────┼──────────────────┼────────────────┤
│ 2019-01 │ Bioenergy        │           0.01 │
│ 2019-01 │ Coal             │            0.0 │
│ 2019-01 │ Gas              │            0.0 │
│ 2019-01 │ Hydro            │           0.28 │
│ 2019-01 │ Other fossil     │            0.1 │
│ 2019-01 │ Other renewables │           0.42 │
│ 2019-01 │ Solar            │           0.01 │
│ 2019-01 │ Total generation │           0.97 │
│ 2019-01 │ Wind             │           0.15 │
│ 2019-02 │ Bioenergy        │           0.01 │
│    ·    │  ·               │             ·  │
│    ·    │  ·               │             ·  │
│    ·    │  ·               │             ·  │
│ 2024-11 │ Wind             │           0.16 │
│ 2024-12 │ Bioenergy        │           0.03 │
│ 2024-12 │ Coal             │            0.0 │
│ 2024-12 │ Gas              │          

In [15]:
power_gen_conn.sql("""
CREATE TABLE energy_mix_per_year(
year INTEGER,
wind_solar_other_renewables DOUBLE,
Bioenergy DOUBLE,
Coal DOUBLE,
Gas_other_fossil_fuels DOUBLE,
Hydro DOUBLE
                   )
""")

CatalogException: Catalog Error: Table with name "energy_mix_per_year" already exists!

In [ ]:
power_gen_conn.sql("SELECT * FROM energy_mix_per_year").show()

┌─────────────────────────────┬───────────┬────────┬────────────────────────┬────────┐
│ wind_solar_other_renewables │ Bioenergy │  Coal  │ Gas_other_fossil_fuels │ Hydro  │
│           double            │  double   │ double │         double         │ double │
├─────────────────────────────┴───────────┴────────┴────────────────────────┴────────┤
│                                       0 rows                                       │
└────────────────────────────────────────────────────────────────────────────────────┘



In [ ]:
power_gen_conn.sql("ALTER TABLE energy_mix_per_year ADD COLUMN year INTEGER")

In [ ]:
power_gen_conn.sql("SELECT * FROM energy_mix_per_year").show()

┌─────────────────────────────┬───────────┬────────┬────────────────────────┬────────┬───────┐
│ wind_solar_other_renewables │ Bioenergy │  Coal  │ Gas_other_fossil_fuels │ Hydro  │ year  │
│           double            │  double   │ double │         double         │ double │ int32 │
├─────────────────────────────┴───────────┴────────┴────────────────────────┴────────┴───────┤
│                                           0 rows                                           │
└────────────────────────────────────────────────────────────────────────────────────────────┘



In [59]:
power_gen_conn.sql("""SELECT generation_twh 
                   FROM energy_mix
                   WHERE generation_twh LIKE 'N%' """)

BinderException: Binder Error: No function matches the given name and argument types '~~(DOUBLE, STRING_LITERAL)'. You might need to add explicit type casts.
	Candidate functions:
	~~(VARCHAR, VARCHAR) -> BOOLEAN


In [28]:
def energy_mix_per_year_data(year,series):
    result=power_gen_conn.sql(f"""
                       SELECT * 
                       FROM energy_mix
                       WHERE date LIKE '{year}%' AND series LIKE '{series}'

                       """).fetchone()
    result=str(result)
    if result=='None':
        result='0'
    k=''
    for i in result:
        if i=='(':
            continue
        if i==',':
            continue
        if i==')':
            continue
        if i=='.':
            k=k+i
            continue
        elif i=='None':
            k=k+'0'
        else:
            k=k+i
    return float(k)

t=energy_mix_per_year_data('2024','Solar')
t

ValueError: could not convert string to float: "'2024-01' 'Solar' 0.04"

In [31]:
print(float(k))

3.0200000000000005


In [97]:

year=2024
all=[]
series=['Bioenergy','Coal','Gas','Hydro','Other fossil','Other renewables','Solar','Wind']
for i in series:
    r=energy_mix_per_year_data(str(year),i)
    all.append(r)
print(all)
print(sum(all))
r=energy_mix_per_year_data(str(year),'Total generation')
print(f"this is the total {r}")
'''
    power_gen_conn.sql(f"""
INSERT INTO energy_mix_per_year (wind_solar_other_renewables,Bioenergy,Coal,Gas_other_fossil_fuels,Hydro,year)
VALUES(({all[-1]+all[-2]+all[-3]}),({all[0]}),({all[1]}),({all[2]+all[4]}),({all[3]}), ({year}))
""")
'''
   # year=+1

[0.27, 0.0, 0.0, 3.6200000000000006, 0.85, 5.55, 0.45999999999999996, 1.8099999999999996]
12.56
this is the total 12.56


'\n    power_gen_conn.sql(f"""\nINSERT INTO energy_mix_per_year (wind_solar_other_renewables,Bioenergy,Coal,Gas_other_fossil_fuels,Hydro,year)\nVALUES(({all[-1]+all[-2]+all[-3]}),({all[0]}),({all[1]}),({all[2]+all[4]}),({all[3]}), ({year}))\n""")\n'

In [ ]:
#deleting coal,gas and total generation records
#power_gen_conn.sql("DELETE FROM energy_mix WHERE series='Total generation' ")

#renaming other renewables to geothermal


In [29]:
power_gen_conn.sql("SELECT * FROM energy_mix").show()

┌─────────┬──────────────────┬────────────────┐
│  date   │      series      │ generation_twh │
│ varchar │     varchar      │     double     │
├─────────┼──────────────────┼────────────────┤
│ 2019-01 │ Bioenergy        │           0.01 │
│ 2019-01 │ Hydro            │           0.28 │
│ 2019-01 │ Other fossil     │            0.1 │
│ 2019-01 │ Other renewables │           0.42 │
│ 2019-01 │ Solar            │           0.01 │
│ 2019-01 │ Wind             │           0.15 │
│ 2019-02 │ Bioenergy        │           0.01 │
│ 2019-02 │ Hydro            │           0.25 │
│ 2019-02 │ Other fossil     │           0.09 │
│ 2019-02 │ Other renewables │           0.37 │
│    ·    │      ·           │             ·  │
│    ·    │      ·           │             ·  │
│    ·    │      ·           │             ·  │
│ 2024-11 │ Other fossil     │           0.08 │
│ 2024-11 │ Other renewables │            0.5 │
│ 2024-11 │ Solar            │           0.04 │
│ 2024-11 │ Wind             │          

In [30]:
import duckdb

# Connect to your DuckDB database
power_gen_conn = duckdb.connect("power_generation.db")

# Set your table name
table_name = 'energy_mix'

print("="*70)
print("ENERGY MIX DATA ANALYSIS")
print("="*70)

# First, let's verify the data exists and see its structure
print("\n📊 Checking existing table structure...")
try:
    # Check available tables
    tables = power_gen_conn.execute("SHOW TABLES").fetchdf()
    print("Available tables in power_generation.db:")
    print(tables)
    print("\n")
    
    # Show ALL original data (no limit)
    print(f"Complete original data from '{table_name}' table:")
    all_data = power_gen_conn.execute(f"SELECT * FROM {table_name} ORDER BY date, series").fetchdf()
    print(all_data.to_string(index=False))
    print(f"\nTotal rows: {len(all_data)}")
    print("\n")
    
    # Check unique series values
    series_list = power_gen_conn.execute(f"SELECT DISTINCT series FROM {table_name} ORDER BY series").fetchdf()
    print("Energy sources found in energy_mix:")
    for idx, series in enumerate(series_list['series'].tolist(), 1):
        print(f"  {idx}. {series}")
    print("\n")
    
    # Check date range (handling YYYY-MM format)
    date_range = power_gen_conn.execute(f"""
        SELECT 
            MIN(date) as earliest_date,
            MAX(date) as latest_date
        FROM {table_name}
    """).fetchdf()
    print(f"Date range: {date_range['earliest_date'][0]} to {date_range['latest_date'][0]}")
    
    # Check total records
    record_count = power_gen_conn.execute(f"SELECT COUNT(*) as count FROM {table_name}").fetchdf()
    print(f"Total records: {record_count['count'][0]:,}")
    
    # Show all unique dates
    unique_dates = power_gen_conn.execute(f"""
        SELECT DISTINCT date 
        FROM {table_name} 
        ORDER BY date
    """).fetchdf()
    print(f"\nUnique dates ({len(unique_dates)} months):")
    print(unique_dates.to_string(index=False))
    print("\n")
    
except Exception as e:
    print(f"❌ Error accessing table 'energy_mix': {e}")
    print("Please make sure:")
    print("  - The database file 'power_generation.db' exists")
    print("  - The table 'energy_mix' exists in the database")
    print("  - The table has the expected columns (date, series, generation_twh)")
    exit(1)

# Get all unique series for dynamic column creation
unique_series = power_gen_conn.execute(f"SELECT DISTINCT series FROM {table_name} ORDER BY series").fetchdf()['series'].tolist()
print(f"🔍 Creating yearly summary with {len(unique_series)} energy sources as columns...")

# Method 1: Using PIVOT with year extraction from YYYY-MM format
print("  - Using PIVOT method...")
power_gen_conn.execute(f"""
CREATE OR REPLACE TABLE yearly_energy_summary AS
PIVOT (
    SELECT 
        SUBSTRING(date, 1, 4) as year,
        series,
        SUM(generation_twh) as total
    FROM {table_name}
    GROUP BY SUBSTRING(date, 1, 4), series
)
ON series
USING SUM(total)
ORDER BY year
""")

# Method 2: Using CASE statements with year extraction
print("  - Using CASE statements method...")
case_statements = []
for series in unique_series:
    # Clean column name (replace spaces and special chars with underscores)
    col_name = series.lower().replace(' ', '_').replace('-', '_').replace('&', 'and')
    case_statements.append(f"    SUM(CASE WHEN series = '{series}' THEN generation_twh ELSE 0 END) as {col_name}")

case_sql = ",\n".join(case_statements)

power_gen_conn.execute(f"""
CREATE OR REPLACE TABLE yearly_energy_summary_cased AS
SELECT 
    SUBSTRING(date, 1, 4) as year,
{case_sql},
    SUM(generation_twh) as total_generation
FROM {table_name}
GROUP BY SUBSTRING(date, 1, 4)
ORDER BY year
""")

# Create a monthly summary table (all months)
print("  - Creating monthly summary table...")
power_gen_conn.execute(f"""
CREATE OR REPLACE TABLE monthly_energy_summary AS
PIVOT (
    SELECT 
        date,
        series,
        generation_twh
    FROM {table_name}
)
ON series
USING SUM(generation_twh)
ORDER BY date
""")

# Create a quarterly summary table
print("  - Creating quarterly summary table...")
power_gen_conn.execute(f"""
CREATE OR REPLACE TABLE quarterly_energy_summary AS
WITH quarterly_data AS (
    SELECT 
        SUBSTRING(date, 1, 4) || '-Q' || ((CAST(SUBSTRING(date, 6, 2) AS INTEGER) - 1) / 3 + 1) as quarter,
        series,
        SUM(generation_twh) as total
    FROM {table_name}
    GROUP BY quarter, series
)
PIVOT quarterly_data ON series USING SUM(total)
ORDER BY quarter
""")

# Display the yearly results (all years)
print("\n" + "="*70)
print("📈 YEARLY ENERGY SUMMARY (All years):")
print("="*70)
yearly_data = power_gen_conn.execute("SELECT * FROM yearly_energy_summary ORDER BY year").fetchdf()
print(yearly_data.to_string(index=False))
print(f"\nTotal years: {len(yearly_data)}")
print("\n")

# Display all monthly data
print("📅 MONTHLY ENERGY SUMMARY (All months):")
print("="*70)
monthly_data = power_gen_conn.execute("SELECT * FROM monthly_energy_summary ORDER BY date").fetchdf()
print(monthly_data.to_string(index=False))
print(f"\nTotal months: {len(monthly_data)}")
print("\n")

# Display quarterly data
print("📊 QUARTERLY ENERGY SUMMARY (All quarters):")
print("="*70)
quarterly_data = power_gen_conn.execute("SELECT * FROM quarterly_energy_summary ORDER BY quarter").fetchdf()
print(quarterly_data.to_string(index=False))
print(f"\nTotal quarters: {len(quarterly_data)}")
print("\n")

# Calculate year-over-year growth (all years)
print("📊 YEAR-OVER-YEAR GROWTH RATES (%):")
print("="*70)

# Build dynamic growth query
growth_columns = []
source_columns = [col for col in yearly_data.columns if col != 'year']

for col in source_columns:
    growth_columns.append(f"    ROUND(100.0 * (curr.{col} - prev.{col}) / NULLIF(prev.{col}, 0), 2) as {col}_growth")

if growth_columns:
    growth_sql = ",\n".join(growth_columns)
    
    growth_query = f"""
        WITH yearly AS (
            SELECT * FROM yearly_energy_summary
        )
        SELECT 
            curr.year,
    {growth_sql}
        FROM yearly curr
        LEFT JOIN yearly prev ON CAST(prev.year AS INTEGER) = CAST(curr.year AS INTEGER) - 1
        WHERE prev.year IS NOT NULL
        ORDER BY curr.year
    """
    
    try:
        growth_data = power_gen_conn.execute(growth_query).fetchdf()
        print(growth_data.to_string(index=False))
    except Exception as e:
        print(f"Note: Could not calculate growth rates: {e}")
else:
    print("No source columns found for growth calculation")
print("\n")

# Calculate total renewable percentage (all years)
print("🌱 RENEWABLE ENERGY PERCENTAGE BY YEAR:")
print("="*70)

# Try to identify renewable sources (customize this list based on your actual data)
renewable_keywords = ['solar', 'wind', 'hydro', 'bio', 'renewable']
renewable_cols = []

for col in yearly_data.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in renewable_keywords):
        renewable_cols.append(col)

if renewable_cols and 'total_generation' in yearly_data.columns:
    renewable_sum = " + ".join(renewable_cols)
    try:
        renewable_pct = power_gen_conn.execute(f"""
            SELECT 
                year,
                {', '.join([f'ROUND({col}, 2) as {col}' for col in renewable_cols])},
                ROUND({renewable_sum}, 2) as total_renewable,
                ROUND(100.0 * ({renewable_sum}) / total_generation, 2) as renewable_percentage
            FROM yearly_energy_summary
            ORDER BY year
        """).fetchdf()
        print(renewable_pct.to_string(index=False))
    except Exception as e:
        print(f"Could not calculate renewable percentage: {e}")
else:
    print("No renewable sources identified or total_generation column missing")
print("\n")

# Export ALL data to CSV (no limits)
print("💾 Exporting ALL data to CSV files...")
power_gen_conn.execute("COPY yearly_energy_summary TO 'yearly_energy_summary.csv' (HEADER, DELIMITER ',')")
power_gen_conn.execute("COPY yearly_energy_summary_cased TO 'yearly_energy_summary_cased.csv' (HEADER, DELIMITER ',')")
power_gen_conn.execute("COPY monthly_energy_summary TO 'monthly_energy_summary.csv' (HEADER, DELIMITER ',')")
power_gen_conn.execute("COPY quarterly_energy_summary TO 'quarterly_energy_summary.csv' (HEADER, DELIMITER ',')")

# Also export the original data
power_gen_conn.execute(f"COPY {table_name} TO 'original_energy_mix.csv' (HEADER, DELIMITER ',')")

print("\n" + "="*70)
print("✅ SUCCESS! Summary tables created in power_generation.db")
print("="*70)
print("\nNew tables created:")
print("  📋 yearly_energy_summary - Yearly summary (all years)")
print("  📋 yearly_energy_summary_cased - Yearly summary (using CASE)")
print("  📋 monthly_energy_summary - Monthly summary (all months)")
print("  📋 quarterly_energy_summary - Quarterly summary (all quarters)")
print("\nCSV files exported:")
print("  📄 yearly_energy_summary.csv")
print("  📄 yearly_energy_summary_cased.csv")
print("  📄 monthly_energy_summary.csv")
print("  📄 quarterly_energy_summary.csv")
print("  📄 original_energy_mix.csv")

# Show table structures
print("\n📋 Column structure of yearly_energy_summary:")
columns = power_gen_conn.execute("DESCRIBE yearly_energy_summary").fetchdf()
for _, row in columns.iterrows():
    print(f"  - {row['column_name']}: {row['column_type']}")

# Show comprehensive summary statistics
print("\n📊 Comprehensive Summary Statistics:")
stats = power_gen_conn.execute("""
    SELECT 
        COUNT(DISTINCT year) as years_covered,
        MIN(year) as first_year,
        MAX(year) as last_year,
        SUM(total_generation) as grand_total_twh,
        ROUND(AVG(total_generation), 2) as avg_yearly_twh,
        ROUND(MIN(total_generation), 2) as min_yearly_twh,
        ROUND(MAX(total_generation), 2) as max_yearly_twh
    FROM yearly_energy_summary_cased
""").fetchdf()
print(stats.to_string(index=False))

# Show top producing year
print("\n🏆 Top Producing Year:")
top_year = power_gen_conn.execute("""
    SELECT 
        year,
        ROUND(total_generation, 2) as total_twh
    FROM yearly_energy_summary_cased
    ORDER BY total_generation DESC
    LIMIT 1
""").fetchdf()
print(top_year.to_string(index=False))

# Show each source's contribution
print("\n⚡ Total Generation by Source (All Time):")
source_totals = power_gen_conn.execute(f"""
    SELECT 
        series,
        ROUND(SUM(generation_twh), 2) as total_twh,
        ROUND(100.0 * SUM(generation_twh) / (SELECT SUM(generation_twh) FROM {table_name}), 2) as percentage
    FROM {table_name}
    GROUP BY series
    ORDER BY total_twh DESC
""").fetchdf()
print(source_totals.to_string(index=False))

# Close connection
power_gen_conn.close()

print("\n👋 Analysis complete! Connection closed.")
print(f"Total files exported: 5 CSV files with complete data")

ENERGY MIX DATA ANALYSIS

📊 Checking existing table structure...
Available tables in power_generation.db:
                  name
0           energy_mix
1  energy_mix_per_year


Complete original data from 'energy_mix' table:
   date           series  generation_twh
2019-01        Bioenergy            0.01
2019-01            Hydro            0.28
2019-01     Other fossil            0.10
2019-01 Other renewables            0.42
2019-01            Solar            0.01
2019-01             Wind            0.15
2019-02        Bioenergy            0.01
2019-02            Hydro            0.25
2019-02     Other fossil            0.09
2019-02 Other renewables            0.37
2019-02            Solar            0.01
2019-02             Wind            0.15
2019-03        Bioenergy            0.01
2019-03            Hydro            0.28
2019-03     Other fossil            0.09
2019-03 Other renewables            0.45
2019-03            Solar            0.01
2019-03             Wind            0